# 07 — Conclusions & feuille de route

**Ce carnet synthétise tout le raisonnement de l'étude et fixe les priorités suivantes.**

---

## Récapitulatif du chemin parcouru

| Carnet | Question | Réponse principale |
|---|---|---|
| 01 | Quelles données ? | WASDE domine, COT utile depuis 2013, météo saisonnière |
| 02 | Saisonnalité ? | Oui, forte — été = volatilité max, hiver = range stable |
| 03 | Facteurs SHAP ? | WASDE exports 15%, saisonnalité 9.5%, SMA-50 7.1% |
| 04 | Modèles vs. baselines ? | H5/H10 : RF gagne. H20/H30 : baseline saisonnière gagne |
| 05 | Incertitude (CQR) ? | 91.7% couverture ✅ — mais intervalles larges |
| 06 | Régimes + backtest ? | Régimes Markov peu exploitables — DCA mensuel > modèle |

---

## Ce qu'on a vraiment prouvé

### ✅ Points validés

1. **Le WASDE est la variable centrale** — 15% de l'importance à h=20j
2. **LightGBM donne un signal directionnel à h=20j** — DA = 61.3% > 55% des baselines
3. **CQR fonctionne** — 91.7% de couverture sur 10 ans de walk-forward
4. **L'anti-leakage est critique** — sans `shift(1)`, les résultats seraient gonflés artificiellement
5. **Le DCA mensuel est robuste** — 4.30 USD/bu, bat la récolte 7 ans sur 10

### ❌ Points non encore prouvés

1. **Les modèles ML ne battent pas les baselines en RMSE à h≥20j**
2. **Le conseil modèle n'améliore pas le revenu vs. DCA simple**
3. **Les régimes Markov ne sont pas exploitables** (3 jours bear/25 ans)
4. **Les sources météo/Crop Progress/FAS ne sont pas encore actives**

---

## Ce que ça explique sur les choix de modélisation

### Pourquoi on a choisi Random Forest comme modèle de base

- Résistant au surapprentissage sans tuning
- Interprétable via SHAP
- Gère les NaN implicitement (pas besoin d'imputation complexe)
- Compatible avec CQR (quantile forest)

### Pourquoi on a choisi LightGBM pour la direction

- Meilleure DA à h=20j (61.3%) grâce aux interactions non-linéaires
- Rapide à entraîner en walk-forward
- Hyperparamètres robustes par défaut

### Pourquoi on n'a pas utilisé les LSTM/deep learning

- 6 192 observations = trop peu pour les réseaux profonds
- Interprétabilité quasi-nulle → non compatible avec conseil agriculteur
- Les modèles à arbres avec facteurs économiques sont compétitifs

### Pourquoi on a construit des facteurs

- 249 features brutes → multicolinéarité sévère
- `others` à 79% dans les résultats SHAP = signal que la factorisation est incomplète
- Objectif : 20-40 facteurs interprétables, chacun avec une signification économique claire

---

## Feuille de route prioritaire

### Phase A — Stabilisation (immédiat)
- [ ] Corriger la factorisation : réduire `others` sous 10%
- [ ] Unifier les artefacts backtest (une seule source officielle)
- [ ] Corriger les régimes Markov (2 états ou rule-based)
- [ ] Améliorer les métriques directionnelles (DA non-neutre, AUC)

### Phase B — Données manquantes (1-2 semaines)
- [ ] Crop Progress NASS (avancement cultural hebdomadaire)
- [ ] Drought Monitor (sécheresse US)
- [ ] FAS Export Sales (volumes d'export hebdomadaires)
- [ ] Prix cash local / basis dynamique

### Phase C — Amélioration modèles (2-3 semaines)
- [ ] Classifier directionnel séparé (classification binaire hausse/baisse)
- [ ] Calibration des probabilités (Platt scaling)
- [ ] CQR par régime et par saison
- [ ] Optuna avec 50+ trials

### Phase D — Backtest réaliste (2 semaines)
- [ ] Basis dynamique (pas fixe à -0.20)
- [ ] Stratégie par prix cible (pas seulement direction)
- [ ] Simulation multi-exploitation
- [ ] Rapport final honnête

In [ ]:
import sys
sys.path.insert(0, '../../src')
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

ROOT = Path('../../')
plt.style.use('seaborn-v0_8-whitegrid')

ss = json.loads((ROOT / 'artefacts/professional_study/study_summary.json').read_text())
bm = pd.read_parquet(ROOT / 'artefacts/professional_study/model_benchmarks.parquet')
bk_sm = json.loads((ROOT / 'artefacts/farmer_backtest/summary.json').read_text())

## Dashboard de synthèse

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.35)

# --- 1. Meilleur RMSE par horizon ---
ax1 = fig.add_subplot(gs[0, 0])
bm_best = bm.groupby('horizon')['rmse'].min().reset_index()
bm_base = bm[bm['model'].str.startswith('baseline')].groupby('horizon')['rmse'].min().reset_index()
ax1.plot(bm_best['horizon'], bm_best['rmse'], 'bo-', label='Meilleur ML', lw=2)
ax1.plot(bm_base['horizon'], bm_base['rmse'], 'r^--', label='Meilleure baseline', lw=2)
ax1.set_title('RMSE : ML vs. baseline', fontsize=10)
ax1.set_xlabel('Horizon (j)')
ax1.set_ylabel('RMSE')
ax1.legend(fontsize=8)

# --- 2. Meilleure DA par horizon ---
ax2 = fig.add_subplot(gs[0, 1])
if 'directional_accuracy' in bm.columns:
    ml_only = bm[~bm['model'].str.startswith('baseline')]
    da_best = ml_only.groupby('horizon')['directional_accuracy'].max().reset_index()
    ax2.bar(da_best['horizon'].astype(str), da_best['directional_accuracy'] * 100,
            color='steelblue', alpha=0.85)
    ax2.axhline(50, color='red', lw=1.5, ls='--')
    ax2.axhline(60, color='green', lw=1, ls=':')
    ax2.set_title('Meilleure DA ML par horizon', fontsize=10)
    ax2.set_ylabel('DA (%)')
    ax2.set_ylim(45, 70)

# --- 3. CQR Coverage ---
ax3 = fig.add_subplot(gs[0, 2])
cqr_df = pd.DataFrame(ss.get('cqr_summary', []))
if not cqr_df.empty:
    ax3.bar(cqr_df['horizon'].astype(str), cqr_df['coverage'] * 100,
            color='#5cb85c', alpha=0.85)
    ax3.axhline(90, color='red', lw=2, ls='--')
    ax3.set_title('CQR Coverage (objectif ≥90%)', fontsize=10)
    ax3.set_ylabel('%')
    ax3.set_ylim(85, 95)

# --- 4. Backtest stratégies ---
ax4 = fig.add_subplot(gs[1, :2])
strategies = bk_sm.get('strategies', [])
if strategies:
    strat_df = pd.DataFrame(strategies)
    colors = ['#d9534f' if 'harvest' in s else ('#5bc0de' if 'dca' in s else '#5cb85c')
              for s in strat_df['strategy']]
    ax4.bar(range(len(strat_df)), strat_df['avg_revenue_per_bu'], color=colors, alpha=0.85)
    ax4.set_xticks(range(len(strat_df)))
    ax4.set_xticklabels(strat_df['strategy'], rotation=15, ha='right', fontsize=9)
    ax4.set_title('Backtest agriculteur — revenu moyen USD/bu (2015-2025, Iowa)', fontsize=10)
    ax4.set_ylabel('USD/bu')
    ax4.set_ylim(3.9, max(strat_df['avg_revenue_per_bu']) * 1.05)

# --- 5. Données actives ---
ax5 = fig.add_subplot(gs[1, 2])
n_active = ss.get('active_sources', 0)
n_planned = ss.get('planned_sources', 0)
n_rows = ss.get('n_rows', 0)
n_factors = ss.get('n_factors', 0)
summary_text = (
    f"RÉSUMÉ\n"
    f"{'─'*25}\n"
    f"Période : 2000-2025\n"
    f"Observations : {n_rows:,}\n"
    f"Facteurs : {n_factors}\n"
    f"Sources actives : {n_active}\n"
    f"Sources planifiées : {n_planned}\n"
    f"CQR moyen : {ss.get('cqr_mean_coverage', 0):.1%}\n"
    f"DA h20 (lgbm) : 61.3%\n"
    f"DCA vs récolte : +0.14$/bu"
)
ax5.text(0.1, 0.5, summary_text, transform=ax5.transAxes,
         fontsize=10, verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax5.axis('off')
ax5.set_title('Métriques clés', fontsize=10)

# --- 6. Signaux manquants ---
ax6 = fig.add_subplot(gs[2, :])
items = [
    ('Crop Progress (NASS)', 0.20, 'Critique'),
    ('Drought Monitor', 0.35, 'Critique'),
    ('FAS Export Sales', 0.30, 'Important'),
    ('Basis cash local', 0.40, 'Important'),
    ('Facteurs propres (others<10%)', 0.55, 'Important'),
    ('Classifier directionnel', 0.45, 'Amélioration'),
    ('CQR par régime/saison', 0.25, 'Amélioration'),
    ('Optuna 50+ trials', 0.15, 'Optionnel'),
]
colors_map = {'Critique': '#F44336', 'Important': '#FF9800', 'Amélioration': '#2196F3', 'Optionnel': '#9E9E9E'}
for i, (label, gain, priority) in enumerate(items):
    ax6.barh(i, gain, color=colors_map[priority], alpha=0.8)
    ax6.text(gain + 0.01, i, f'{priority}', va='center', fontsize=8)
ax6.set_yticks(range(len(items)))
ax6.set_yticklabels([x[0] for x in items], fontsize=9)
ax6.set_title('Prochaines améliorations — gain estimé sur DA (impact relatif)', fontsize=10)
ax6.set_xlabel('Impact estimé (relatif)')
ax6.set_xlim(0, 0.7)
from matplotlib.patches import Patch
legend = [Patch(facecolor=c, label=l) for l, c in colors_map.items()]
ax6.legend(handles=legend, loc='lower right', fontsize=8)

plt.suptitle('Etude Maïs CBOT — Tableau de bord de synthèse', fontsize=15, y=1.01)
plt.savefig(ROOT / 'docs/study_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard sauvegardé dans docs/study_dashboard.png')

## Décision courante (signal du jour)

In [ ]:
dec = ss.get('decision', {})
if dec and dec.get('status') == 'ok':
    rec = dec.get('recommendation', {})
    print(f"""Signal du {dec.get('as_of', '—')}
{'='*50}
Prix CBOT    : {dec.get('raw_cbot_quote')} cts/bu
Cash Iowa    : {dec.get('cash_price_usd_per_bu')} USD/bu
Régime       : {dec.get('regime')}

Intervalle 20j :
  Q10 = {dec.get('predicted_cash_q10_h20', '—'):.3f} USD/bu
  Q50 = {dec.get('predicted_cash_q50_h20', '—'):.3f} USD/bu  
  Q90 = {dec.get('predicted_cash_q90_h20', '—'):.3f} USD/bu

Action       : {rec.get('action')}
Fraction     : {rec.get('sell_fraction', 0):.0%}
Règle        : {rec.get('rule_id')}
Raison       : {rec.get('rationale')}
""")